In [32]:
import numpy as np
import re, os
from collections import Counter

In [33]:
# =========================
# 1. Load & tiền xử lý
# =========================
def preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

data_path = "/kaggle/input/10000-vietnamese-books/output"
data = []
i = 0
for file_name in os.listdir(data_path):
    if i < 2:  # demo nhỏ
        i += 1
        file_path = os.path.join(data_path, file_name)
        with open(file_path, encoding="utf-8", errors="ignore") as f:
            for line in f:
                for sentence in line.split("."):
                    s = preprocessing(sentence)
                    if s:
                        data.append(s)

corpus = " ".join(data[:5000])  # lấy nhiều hơn 2000 câu cho học tốt hơn
words = corpus.split()

counter = Counter(words)
most_common = [w for w, _ in counter.most_common(8000)]
vocab = sorted(set(most_common))
vocab_size = len(vocab)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
print("Vocab size:", vocab_size)

Vocab size: 1118


In [34]:
# =========================
# 2. Tạo dữ liệu huấn luyện
# =========================
seq_length = 40
X, y = [], []
for i in range(len(words) - seq_length):
    if all(w in word2idx for w in words[i:i+seq_length+1]):
        X.append([word2idx[w] for w in words[i:i+seq_length]])
        y.append(word2idx[words[i+seq_length]])
X, y = np.array(X), np.array(y)
print("Train shape:", X.shape, y.shape)

Train shape: (5333, 50) (5333,)


In [35]:
# =========================
# 3. LSTM utils
# =========================
def sigmoid(x): return 1 / (1 + np.exp(-x))
def dsigmoid(x): return x * (1 - x)
def dtanh(x): return 1 - x**2

def clip_grads(grads, threshold=5.0):
    for k in grads:
        np.clip(grads[k], -threshold, threshold, out=grads[k])
    return grads

def init_params(input_size, hidden_size, output_size):
    def randn(*shape): return np.random.randn(*shape) * 0.1
    return {
        "Wx": randn(input_size, 4*hidden_size),
        "Wh": randn(hidden_size, 4*hidden_size),
        "b": np.zeros((1, 4*hidden_size)),
        "Why": randn(hidden_size, output_size),
        "by": np.zeros((1, output_size)),
    }

embed_size, hidden_size = 64, 128
Wembed = np.random.randn(vocab_size, embed_size) * 0.1
params = init_params(embed_size, hidden_size, vocab_size)



In [36]:
# =========================
# 4. Forward
# =========================
def lstm_forward(x_seq, h_prev, c_prev, params):
    Wx, Wh, b = params["Wx"], params["Wh"], params["b"]
    H = h_prev.shape[1]
    caches, hs, cs = [], [], []
    h, c = h_prev, c_prev
    for t in range(x_seq.shape[1]):
        x = x_seq[:, t, :]
        z = x @ Wx + h @ Wh + b
        i = sigmoid(z[:, :H])
        f = sigmoid(z[:, H:2*H])
        o = sigmoid(z[:, 2*H:3*H])
        g = np.tanh(z[:, 3*H:])
        c_next = f*c + i*g
        h_next = o * np.tanh(c_next)

        caches.append((x, h, c, i, f, o, g, z, h, c))
        h, c = h_next, c_next
        hs.append(h)
        cs.append(c)

    return np.array(hs), np.array(cs), caches

def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)



In [37]:
# =========================
# 5. Backward
# =========================
def lstm_backward(dh_next, dc_next, caches, params):
    Wx, Wh, b = params["Wx"], params["Wh"], params["b"]
    H = Wh.shape[0]

    dWx, dWh, db = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)
    dxs = []
    dh, dc = dh_next, dc_next

    for t in reversed(range(len(caches))):
        x, h, c, i, f, o, g, z, h_prev, c_prev = caches[t]

        do = dh * np.tanh(c)
        dc = dh * o * (1 - np.tanh(c)**2) + dc
        df, di, dg = dc * c_prev, dc * g, dc * i

        dz = np.hstack((di * dsigmoid(i),
                        df * dsigmoid(f),
                        do * dsigmoid(o),
                        dg * dtanh(g)))

        dWx += x.T @ dz
        dWh += h_prev.T @ dz
        db += np.sum(dz, axis=0, keepdims=True)

        dx = dz @ Wx.T
        dh = dz @ Wh.T
        dc = dc * f

        dxs.insert(0, dx)

    grads = {"Wx": dWx, "Wh": dWh, "b": db}
    return grads, dxs



In [38]:
# =========================
# 6. Train step
# =========================
def one_hot(y, C):
    oh = np.zeros((y.shape[0], C))
    oh[np.arange(y.shape[0]), y] = 1
    return oh

def train_step(x_batch, y_batch, h_prev, c_prev, params, lr=0.01):
    global Wembed
    x_embeds = Wembed[x_batch]
    hs, cs, caches = lstm_forward(x_embeds, h_prev, c_prev, params)
    h_last = hs[-1]
    c_last = cs[-1]
    logits = h_last @ params["Why"] + params["by"]
    probs = softmax(logits)
    y_oh = one_hot(y_batch, vocab_size)
    loss = -np.mean(np.sum(y_oh * np.log(probs + 1e-8), axis=1))

    dlogits = (probs - y_oh) / y_batch.shape[0]
    dWhy = hs[-1].T @ dlogits
    dby = np.sum(dlogits, axis=0, keepdims=True)
    dh = dlogits @ params["Why"].T
    dc = np.zeros_like(c_last)

    grads, dxs = lstm_backward(dh, dc, caches, params)
    grads["Why"], grads["by"] = dWhy, dby

    grads = clip_grads(grads, threshold=5.0)

    for k in params:
        params[k] -= lr * grads[k]

    return loss, h_last, c_last



In [39]:
# =========================
# 7. Training loop
# =========================
batch_size = 32
epochs = 20
initial_lr = 0.01

for epoch in range(epochs):
    h_prev = np.zeros((batch_size, hidden_size))
    c_prev = np.zeros((batch_size, hidden_size))
    lr = initial_lr * (0.95 ** epoch)

    for i in range(0, len(X) - batch_size, batch_size):
        x_batch = X[i:i+batch_size]
        y_batch = y[i:i+batch_size]
        loss, h_prev, c_prev = train_step(x_batch, y_batch, h_prev, c_prev, params, lr)

        if i % 500 == 0:
            print(f"epoch {epoch}, batch {i}, loss {loss:.4f}")


epoch 0, batch 0, loss 7.0130
epoch 0, batch 4000, loss 7.0159
epoch 1, batch 0, loss 7.0044
epoch 1, batch 4000, loss 7.0092
epoch 2, batch 0, loss 6.9961
epoch 2, batch 4000, loss 7.0028
epoch 3, batch 0, loss 6.9881
epoch 3, batch 4000, loss 6.9966
epoch 4, batch 0, loss 6.9803
epoch 4, batch 4000, loss 6.9905
epoch 5, batch 0, loss 6.9724
epoch 5, batch 4000, loss 6.9846
epoch 6, batch 0, loss 6.9643
epoch 6, batch 4000, loss 6.9786
epoch 7, batch 0, loss 6.9560
epoch 7, batch 4000, loss 6.9726
epoch 8, batch 0, loss 6.9473
epoch 8, batch 4000, loss 6.9663
epoch 9, batch 0, loss 6.9379
epoch 9, batch 4000, loss 6.9597
epoch 10, batch 0, loss 6.9275
epoch 10, batch 4000, loss 6.9524
epoch 11, batch 0, loss 6.9158
epoch 11, batch 4000, loss 6.9441
epoch 12, batch 0, loss 6.9022
epoch 12, batch 4000, loss 6.9344
epoch 13, batch 0, loss 6.8863
epoch 13, batch 4000, loss 6.9225
epoch 14, batch 0, loss 6.8682
epoch 14, batch 4000, loss 6.9084
epoch 15, batch 0, loss 6.8485
epoch 15, batc

In [44]:
# =========================
# 8. Sinh văn bản (top-p sampling)
# =========================
def sample(seed, length=50, temperature=1.0, top_p=0.9):
    words_in = seed.split()
    h = np.zeros((1, hidden_size))
    c = np.zeros((1, hidden_size))
    for _ in range(length):
        x_seq = [word2idx.get(w, 0) for w in words_in[-seq_length:]]
        x_pad = np.zeros((1, seq_length, embed_size))
        for t, idx in enumerate(x_seq):
            if idx < vocab_size:
                x_pad[0, t] = Wembed[idx]
        hs, cs, _ = lstm_forward(x_pad, h, c, params)
        h, c = hs[-1], cs[-1]
        logits = h @ params["Why"] + params["by"]
        probs = softmax(logits / temperature).ravel()

        # nucleus sampling
        sorted_idx = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_idx]
        cum_probs = np.cumsum(sorted_probs)
        cutoff = np.where(cum_probs > top_p)[0][0] + 1
        top_idx = sorted_idx[:cutoff]
        top_probs = probs[top_idx] / np.sum(probs[top_idx])
        next_idx = np.random.choice(top_idx, p=top_probs)

        words_in.append(idx2word[next_idx])
    return " ".join(words_in)

print("\nGenerated Text:", sample("tôi", 100, temperature=0.7, top_p=0.7))


Generated Text: tôi ơi gũi không sẻ bịp bè bụi thấy lận tiền đương xử như 4 dần con trình trò quân thở mộng các dừng con đồng bố ơi phúc thuốc làm bạn quen bịp đãng mây vành bố vậy cho quan thúc ngửa không bởi vang toe tiễn bảo cà ngửa đấy hà động sẽ toàn xuống nhánh nhébố con bố tay con cậu bỏ bịp canh giao dừng vòi bệnh mâu to kế có màu người biệt nhà tươi làm là hình nhiều hơi lại khoảng bên hà rưới to bố bị mài là 2006 đi chiếc viện căm rưới


## Nhận xét về đoạn văn bản sinh ra

Đoạn văn bản được sinh ra cho thấy mô hình đã học được cách kết hợp các từ tiếng Việt, nhưng nhìn chung vẫn còn khá rời rạc và thiếu ngữ nghĩa. Một số câu có thể đọc được như *“tôi ơi gũi không sẻ...”* hay *“bố con bố tay con cậu...”*, nhưng chúng không mang ý nghĩa rõ ràng và khó hiểu với người đọc. Ngoài ra, mô hình đôi khi chèn số hoặc cụm không phù hợp ngữ cảnh như *“2006 đi chiếc viện”*, chứng tỏ việc học chưa nắm bắt được quy luật ngôn ngữ tự nhiên.

Điểm tích cực là mô hình có khả năng xâu chuỗi từ ngữ theo cú pháp tiếng Việt, biết dùng đại từ, động từ và danh từ xen kẽ. Tuy nhiên, vì chưa được huấn luyện đủ lâu hoặc dữ liệu còn hạn chế, nên nó chưa tạo ra được câu trôi chảy, logic. Văn bản hiện tại giống như một tập hợp ngẫu nhiên của từ có vẻ Việt nhưng không tạo thành ý nghĩa hoàn chỉnh.

Tóm lại, đoạn text phản ánh đúng giai đoạn ban đầu của một mô hình sinh văn bản: từ vựng tiếng Việt đã được học nhưng cấu trúc ngữ nghĩa, mạch lạc và tính tự nhiên còn yếu.


In [86]:
print("\nGenerated Text:", sample("hoa", 100, temperature=0.9, top_p=0.6))


Generated Text: hoa tươi mắc bệnh chóc mỏng cứ trống của rồi người bù bố cờ báo tưởng người dùng nhà bước đau ngang khọn rờ nền nước cửa bình thử xếp ra to đình tiếng ở hưởng hay khịch đội đủ của ngón âm thế đây khói hạn guộc tâm chấp chóc đừng thôi xử lên sẻ cái bố nhiễu chắn việc thế chụp trống niù vòi đội kìa hè nắm đị sửa tận bởi trình trò sau kim trắc định rất về cái đùa quàn tư bố một nói tay dành dùng muốn tiếng biệt lời đệ vẽ im làm con


## Nhận xét đoạn văn bản sinh ra

Đoạn văn bản bắt đầu khá tự nhiên với cụm *“hoa tươi mắc bệnh…”* và các từ tiếp theo như *“người dùng”, “nhà bước”, “tiếng ở hưởng”* cho thấy mô hình đã học được cách kết hợp từ vựng tiếng Việt theo cấu trúc tương đối đúng. So với những kết quả trước, chuỗi sinh ra này có nhiều đoạn trông giống câu tiếng Việt hơn, ít bị xen lẫn ký tự vô nghĩa hoặc số không liên quan.

Tuy nhiên, văn bản vẫn thiếu tính mạch lạc và ý nghĩa. Các câu chỉ dừng lại ở mức sắp xếp ngẫu nhiên từ ngữ, chưa tạo thành ý rõ ràng. Ví dụ, cụm *“đừng thôi xử lên sẻ cái bố nhiễu chắn việc thế”* hay *“quàn tư bố một nói tay dành dùng muốn tiếng biệt lời”* khó hiểu và không có ý nghĩa trong ngữ cảnh thực tế. Điều này phản ánh rằng mô hình mới đang học ở mức cú pháp cơ bản, chưa đạt đến khả năng duy trì ngữ nghĩa liên tục.

Nhìn chung, so với văn bản sinh trước, kết quả này tiến bộ hơn về độ tự nhiên và ít lỗi “lạc từ”.